# Download data from Netztransperenz:
## Important: UTC format consider
## I use Chunks do ammount of data
## Data download from this sources:
### ReBAP: https://www.netztransparenz.de/en/Balancing-Capacity/Imbalance-price/Uniform-imbalance-price-reBAP
### Activated AFRR: https://www.netztransparenz.de/en/Balancing-Capacity/Balancing-Capacity-data/Activated-Balancing-Capacity

# STEP 1: Download every 15 min data

In [ ]:
import requests
import pandas as pd
import io
import time

# Credentials
CLIENT_ID     = "cm_app_ntp_id_eb6644b175544f58b34cc41c76c758a6"
CLIENT_SECRET = "ntp_4usaQtsM2pAS23hTB9HG"

TOKEN_URL    = "https://identity.netztransparenz.de/users/connect/token"
BASE_API_URL = "https://ds.netztransparenz.de/api/v1/data"

# Auth
def get_access_token(client_id, client_secret):
    response = requests.post(TOKEN_URL, data={
        'grant_type':    'client_credentials',
        'client_id':     client_id,
        'client_secret': client_secret
    }, headers={'Content-Type': 'application/x-www-form-urlencoded'})
    response.raise_for_status()
    return response.json()['access_token']

# Chunked fetcher
def fetch_chunked(token, data_type, product_type=None, start_date="2022-01-01", end_date="2025-04-19", chunk="year"):
    start = pd.Timestamp(start_date)
    end   = pd.Timestamp(end_date)

    ranges, current = [], start
    while current < end:
        next_chunk = current + pd.DateOffset(years=1 if chunk == "year" else 0, months=0 if chunk == "year" else 1)
        ranges.append((current.strftime("%Y-%m-%d"), min(next_chunk, end).strftime("%Y-%m-%d")))
        current = next_chunk

    all_dfs = []
    for s, e in ranges:
        url = f"{BASE_API_URL}/{data_type}/{product_type}/{s}/{e}" if product_type else f"{BASE_API_URL}/{data_type}/{s}/{e}"
        try:
            response = requests.get(url, headers={"Authorization": f"Bearer {token}"}, timeout=30)
            response.raise_for_status()

            if not response.text.strip():
                print(f"Empty response")
                continue

            df = pd.read_csv(io.StringIO(response.text), sep=";", decimal=",", encoding="utf-8")

            # Formato 9, 6.a, 6.b, 7.a, 7.b, 8, 10, 11, 14, 17, 18
            if "Datum" in df.columns and "von" in df.columns:
                df["date"] = pd.to_datetime(df["Datum"] + " " + df["von"], format="%d.%m.%Y %H:%M")
                df.drop(columns=["Datum", "von", "bis", "Zeitzone", "Datentyp", "Datenkategorie", "Einheit"], errors="ignore", inplace=True)

            # Formato 13: Datum von;(Uhrzeit) von;Zeitzone;(Uhrzeit) bis;Zeitzone;ID AEP in €/MWh
            elif "Datum von" in df.columns and "(Uhrzeit) von" in df.columns:
                df["date"] = pd.to_datetime(df["Datum von"] + " " + df["(Uhrzeit) von"], format="%Y-%m-%d %H:%M")
                df.drop(columns=["Datum von", "(Uhrzeit) von", "(Uhrzeit) bis", "Zeitzone", "Zeitzone von", "Zeitzone bis", "Zeitzone.1"], errors="ignore", inplace=True)

            all_dfs.append(df)
            time.sleep(0.5)

        except Exception as err:
            print(f"Error:{err}")
            continue

    if not all_dfs:
        return pd.DataFrame()

    result = pd.concat(all_dfs, ignore_index=True).sort_values("date").reset_index(drop=True)
    print(f"{len(result)} rows | columns: {list(result.columns)}")
    return result


if __name__ == "__main__":
    start_date = "2022-01-01"
    end_date   = "2026-01-01"

    token = get_access_token(CLIENT_ID, CLIENT_SECRET)

    # --- reBAP ueberdeckt (Formato 9) ---
    # Columnas: reBAP unterdeckt, reBAP ueberdeckt
    df_rebap = fetch_chunked(token, "NrvSaldo/reBAP", "Qualitaetsgesichert", start_date, end_date)
    df_rebap = df_rebap[["date", "reBAP ueberdeckt"]].copy()

    # --- aFRR betrieblich (Formato 6.a) ---
    # Columnas: 50Hertz (Positiv/Negativ), Amprion (Positiv/Negativ),
    #           TenneT TSO (Positiv/Negativ), TransnetBW (Positiv/Negativ),
    #           Deutschland (Positiv/Negativ), MOL-Abweichung
    df_srl = fetch_chunked(token, "NrvSaldo/AktivierteSRL", "Betrieblich", start_date, end_date)
    df_srl.rename(columns={
        "Deutschland (Positiv)": "Deutschland pos",
        "Deutschland (Negativ)": "Deutschland neg",
        "50Hertz (Positiv)":     "50Hertz pos",
        "50Hertz (Negativ)":     "50Hertz neg",
        "Amprion (Positiv)":     "Amprion pos",
        "Amprion (Negativ)":     "Amprion neg",
        "TenneT TSO (Positiv)":  "TenneT TSO pos",
        "TenneT TSO (Negativ)":  "TenneT TSO neg",
        "TransnetBW (Positiv)":  "TransnetBW pos",
        "TransnetBW (Negativ)":  "TransnetBW neg",
        "MOL-Abweichung":        "MOL Abweichung SRL",
    }, inplace=True)

    # merge
    
    merged_df = df_rebap \
        .merge(df_srl, on="date", how="outer") \
        .sort_values("date").reset_index(drop=True)

    target_columns = [
        "date",
        "reBAP ueberdeckt",
        "Deutschland pos", "Deutschland neg",
        "50Hertz pos",     "50Hertz neg",
        "Amprion pos",     "Amprion neg",
        "TenneT TSO pos",  "TenneT TSO neg",
        "TransnetBW pos",  "TransnetBW neg",
        "MOL Abweichung SRL",
    ]

    available = [c for c in target_columns if c in merged_df.columns]
    missing   = [c for c in target_columns if c not in merged_df.columns]

    if missing:
        print(f"missing: {missing}")

    final_df = merged_df[available]
    print(f"\n Final shape: {final_df.shape}")
    print(final_df.head(10))

    final_df.to_csv("Netztransparenz_features_15min.csv", index=False)

## STEP 2: DOWNSAMPLED TO HOURLY DATA

In [ ]:
import pandas as pd

# Load CSV
df = pd.read_csv("netztransparenz_features_15min.csv")

# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Keep only exact hourly timestamps
df_hourly = df[df["date"].dt.minute == 0]

# Save
df_hourly.to_csv("Netztransparenz_2025.csv", index=False)

print(df_hourly.head())